# Classificação Fine-Grained de Raças de Cães

**Trabalho final — Processamento e Análise de Imagens**
Pós-graduação em LLM e IA Generativa

Integrantes: Mariana Zanon e Victor Macaúbas
Data: agosto/2026
Ambiente de treinamento: Google Colab com GPU T4

Repositório: https://github.com/victormacaubas/project-image-processing

---

> **Este notebook é autossuficiente e roda em ~2 minutos.**
>
> Basta `Ambiente de execução → Executar tudo`. A primeira célula clona o
> repositório e instala o que falta; nenhum arquivo adicional é necessário e
> nada precisa ser baixado.
>
> Os resultados, gráficos e a análise de erros são reconstruídos a partir das
> predições (logits) versionadas no repositório — por isso é rápido.
>
> A execução padrão não requer GPU: ela utiliza os artefatos versionados no
> repositório. Os treinamentos que produziram esses resultados foram executados
> no Google Colab com runtime T4 GPU.

In [ ]:
REPO_URL = "https://github.com/victormacaubas/project-image-processing.git"
REPO_NAME = "project-image-processing"

import subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules


def _run(cmd: list[str]) -> None:
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stdout)
        print(result.stderr, file=sys.stderr)
        raise RuntimeError(f"Falhou: {' '.join(cmd)}")


def _find_repo_root() -> Path:
    here = Path.cwd()

    for candidate in (here, *here.parents):
        if (candidate / "src" / "dogs" / "config.py").exists():
            return candidate

    if (here / REPO_NAME / "src" / "dogs" / "config.py").exists():
        return here / REPO_NAME

    print(f"Clonando {REPO_URL} ...")
    _run(["git", "clone", "--depth", "1", REPO_URL, REPO_NAME])
    return here / REPO_NAME


REPO_ROOT = _find_repo_root()

if IN_COLAB:
    reqs = REPO_ROOT / "requirements-colab.txt"
    if reqs.exists():
        print("Instalando dependências ...")
        _run([sys.executable, "-m", "pip", "install", "-q", "-r", str(reqs)])

src = str(REPO_ROOT / "src")
if src not in sys.path:
    sys.path.insert(0, src)

print(f"Repositório: {REPO_ROOT}")

### Verificação do ambiente

A célula a seguir confere as bibliotecas necessárias e prepara os diretórios usados pelos resultados.

In [ ]:
import importlib

problemas = []

for pacote in ["torch", "torchvision", "numpy", "pandas", "sklearn",
               "matplotlib", "seaborn", "datasets"]:
    try:
        importlib.import_module(pacote)
    except ImportError as erro:
        problemas.append(f"pacote ausente: {pacote} ({erro})")

try:
    from dogs.config import describe_environment, ensure_dirs
    ensure_dirs()
    print(describe_environment())
except Exception as erro:
    problemas.append(f"pacote `dogs` não importável: {erro}")

if problemas:
    raise RuntimeError(
        "Ambiente incompleto:\n  - " + "\n  - ".join(problemas)
        + "\n\nRode a célula de bootstrap acima antes desta."
    )

print("\nAmbiente OK.")

### Configuração da execução

A execução padrão reconstrói a apresentação a partir dos artefatos versionados. Ela não exige GPU; o runtime **T4 GPU** do Google Colab foi utilizado apenas nos treinamentos que geraram os resultados apresentados.

In [ ]:
RETRAIN = False

RETRAIN_E3 = False

import logging
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt, seaborn as sns

from dogs.config import (TrainConfig, FEATURES_DIR, CHECKPOINT_DIR,
                         PREDICTIONS_DIR, RESULTS_CSV)

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s", force=True)
torch.manual_seed(42)
np.random.seed(42)

---
# 1. Descrição do problema

O objetivo deste trabalho é classificar, a partir de uma fotografia, a raça de um cão entre 120 possibilidades. Trata-se de um problema de *classificação fine-grained*: ao contrário de uma classificação genérica, em que as classes podem ser visualmente distantes (por exemplo, cão, carro e avião), todas as categorias aqui pertencem ao mesmo grupo semântico. A decisão depende, portanto, de evidências sutis como formato do focinho, textura e cor da pelagem, proporção das orelhas e estrutura corporal.

A tarefa é difícil por combinar baixa variância entre classes com alta variância dentro da própria classe. Pose, iluminação, idade do animal, fundo, oclusões e enquadramento podem tornar duas imagens da mesma raça visualmente mais diferentes entre si do que imagens de duas raças próximas. Essa situação aparece em aplicações reais de catalogação de animais, apoio a abrigos e clínicas veterinárias, organização de acervos fotográficos e interfaces de busca por imagem.

> **Pergunta que guia o trabalho: quanto de representação visual é preciso aprender, em vez de transferir, para resolver classificação fine-grained com dados limitados?**

Para respondê-la, comparamos uma CNN pequena treinada do zero, um classificador linear sobre embeddings de uma ResNet50 congelada e o fine-tuning parcial da mesma arquitetura. Consideramos sucesso não apenas obter alta acurácia top-1, mas também mostrar uma progressão coerente entre os experimentos, boa acurácia top-5 e F1 macro compatível com o desempenho global, além de analisar os erros mais informativos.

---
# 2. Descrição da base de dados

Usamos o Stanford Dogs, proposto por Khosla et al. para classificação subordinada de raças. A base reúne 20.580 imagens de 120 raças, com split oficial de 12.000 imagens de treino e 8.580 de teste. A partir do treino oficial, reservamos 15% para validação com semente fixa `SEED = 42`, obtendo 10.200 imagens de treino e 1.800 de validação. O teste permanece separado e é usado somente na avaliação final do melhor modelo. Por derivar de imagens do ImageNet, a utilização segue os termos de pesquisa e educação não comercial do ImageNet; ele não detém os direitos autorais individuais das imagens.

O enunciado indicava `Voxel51/StanfordDogs`, mas esse repositório está no formato FiftyOne: `load_dataset()` não falha, porém retorna as 20.580 imagens em um único split e sem a coluna de rótulos, uma falha silenciosa que só apareceria no treinamento. Por isso adotamos `maurice-fp/stanford-dogs`, versão em parquet que preserva o split oficial, a coluna `label` como `ClassLabel` e os nomes WordNet das raças, como `n02085620-Chihuahua`. Essa escolha foi uma decisão de engenharia de dados para garantir rótulos, reprodutibilidade e validação correta antes de qualquer treinamento.

A Figura 1 mostra uma distribuição perfeitamente balanceada: mínimo, máximo e mediana são todos 100 imagens por raça no treino oficial. Assim, F1 macro não é necessária para compensar desbalanceamento, mas permanece útil como confirmação de que o desempenho não se concentra em poucas classes. A Figura 2 mostra amostras cruas de dez raças, a Figura 3 evidencia a variedade de proporções e resoluções originais que motiva o redimensionamento para 224×224, e a Figura 4 registra pares visualmente próximos escolhidos antes do treino. Como o Stanford Dogs foi construído a partir do ImageNet, essa origem também é uma ameaça importante à validade externa e é discutida na Seção 6.

A célula a seguir apresenta as quatro figuras exploratórias usadas na caracterização da base.

In [ ]:
import json

from IPython.display import Image, display

from dogs.config import FIGURES_DIR, REPORTS_DIR

NOMES_FIGURAS_EDA = [
    "distribuicao_classes",
    "grid_amostras",
    "resolucoes",
    "pares_parecidos",
]

if RETRAIN:
    from dogs.data import _load_raw
    from dogs.eda import (
        dispersao_resolucoes,
        distribuicao_classes,
        grid_pares_parecidos,
        indices_por_classe,
    )
    from dogs.viz import grid_de_amostras

    dataset_raw = _load_raw()
    class_names = list(dataset_raw["train"].features["label"].names)
    assert len(class_names) == 120
    (REPORTS_DIR / "class_names.json").write_text(
        json.dumps(class_names, ensure_ascii=False, indent=0), encoding="utf-8"
    )

    _, estatisticas_classes = distribuicao_classes(
        dataset_raw["train"]["label"],
        class_names,
        salvar_em=FIGURES_DIR / "distribuicao_classes.png",
    )
    print("Imagens por classe — " + ", ".join(
        f"{nome}: {valor:.0f}" for nome, valor in estatisticas_classes.items()
    ))

    classes_amostra = np.linspace(0, len(class_names) - 1, num=10, dtype=int)
    grid_de_amostras(
        dataset_raw["train"],
        indices_por_classe(dataset_raw["train"], classes_amostra),
        class_names,
        n_colunas=4,
        titulo="Amostras de raças do Stanford Dogs",
        salvar_em=FIGURES_DIR / "grid_amostras.png",
    )
    dispersao_resolucoes(
        dataset_raw["train"], salvar_em=FIGURES_DIR / "resolucoes.png"
    )
    pares_apostados = [
        ("n02093754-Border_terrier", "n02095889-Sealyham_terrier"),
        ("n02093991-Irish_terrier", "n02094114-Norfolk_terrier"),
        ("n02110063-malamute", "n02110185-Siberian_husky"),
        ("n02113023-Pembroke", "n02113186-Cardigan"),
    ]
    grid_pares_parecidos(
        dataset_raw["train"],
        class_names,
        pares_apostados,
        salvar_em=FIGURES_DIR / "pares_parecidos.png",
    )
else:
    for nome in NOMES_FIGURAS_EDA:
        caminho = FIGURES_DIR / f"{nome}.png"
        if not caminho.exists():
            raise FileNotFoundError(
                f"Figura da EDA ausente: {caminho.name}."
            )
        display(Image(filename=str(caminho)))

---
# 3. Metodologia

A metodologia progride de uma representação aprendida inteiramente com a base para uma representação transferida e, por fim, adaptada. O Experimento 01 usa uma CNN pequena treinada do zero, formada por quatro blocos `Conv2d(3×3) → BatchNorm2d → ReLU → MaxPool2d(2)`, com 32, 64, 128 e 256 canais. Um `AdaptiveAvgPool2d(1)` reduz a saída a 256 atributos, seguidos de dropout e uma camada linear para as 120 classes. A arquitetura possui 420.216 parâmetros — valor calculado pela própria implementação, em vez da estimativa de aproximadamente um milhão do roteiro.

No Experimento 02, congelamos uma ResNet50 pré-treinada e treinamos apenas `BatchNorm1d → Linear` sobre seus embeddings de 2.048 dimensões. A normalização é habilitada e desabilitada em uma ablação: como as features vêm após uma ReLU, são não negativas e têm escalas distintas entre dimensões; normalizá-las melhora o condicionamento do problema para o classificador linear. No Experimento 03, substituímos a cabeça da ResNet50 e descongelamos somente os dois últimos blocos, permitindo adaptar features de alto nível sem atualizar todo o backbone.

As imagens são convertidas para RGB, redimensionadas e normalizadas com média e desvio-padrão do ImageNet. Apenas o conjunto de treino recebe `RandomResizedCrop`, espelhamento horizontal e pequenas variações de cor; validação e teste usam `Resize` seguido de `CenterCrop`, para que a avaliação seja determinística e não introduza transformações aleatórias. Todos os experimentos usam AdamW, scheduler cosseno, label smoothing de 0,1, precisão mista e early stopping com paciência quatro, preservando o checkpoint de maior top-1 na validação. Os treinos foram executados em Google Colab com GPU T4.

A métrica principal é a acurácia top-1. A top-5 é relevante em 120 classes fine-grained porque informa se a raça correta ficou entre hipóteses visualmente plausíveis; o F1 macro impede que um bom desempenho em classes mais frequentes esconda falhas em classes menores. Para reduzir custo computacional, os embeddings da ResNet50 foram pré-computados uma única vez: a extração sobre as imagens custa minutos em GPU, mas os classificadores lineares posteriores treinam em segundos na CPU. Os Experimentos 01 e 02 são comparados exclusivamente na validação; o split de teste é reservado para uma única avaliação do Experimento 03.

---
# 4. Experimentos

Cada experimento: o que testa, como foi configurado, o que aconteceu.

## Experimento 01 — CNN treinada do zero

**Hipótese:** sem transferência, 120 classes fine-grained com ~100 imagens de treino
por classe não dão sinal suficiente. Esperamos acurácia baixa.

A célula a seguir recupera as predições do Experimento 01 e calcula suas métricas de validação.

In [ ]:
from dogs.evaluate import load_predictions, metrics_from_logits
from dogs.models import SmallCNN

NOME = "E1_scratch"

if RETRAIN:
    from dogs.data import load_data
    from dogs.evaluate import log_result, predict, save_predictions
    from dogs.train import get_device, load_checkpoint, train_model

    config_e1 = TrainConfig(
        experiment_name=NOME, num_epochs=15, learning_rate=3e-4
    )
    model_e1 = SmallCNN()
    saida_sanity = model_e1(torch.randn(2, 3, 224, 224))
    assert saida_sanity.shape == (2, 120)
    print(f"sanidade: {tuple(saida_sanity.shape)}, "
          f"{sum(p.numel() for p in model_e1.parameters()):,} parâmetros")

    dados_e1 = load_data(config_e1)
    print(train_model(model_e1, dados_e1.train_loader, dados_e1.val_loader, config_e1))
    model_e1 = load_checkpoint(model_e1, config_e1)
    logits, labels = predict(model_e1, dados_e1.val_loader, get_device())
    metricas_e1 = metrics_from_logits(logits, labels)
    log_result(NOME, "val", metricas_e1)
    save_predictions(NOME, "val", logits, labels)
    print(f"Experimento 01 — validação: {metricas_e1}")
else:
    logits, labels = load_predictions(NOME, "val")
    print(f"Experimento 01 — validação: {metrics_from_logits(logits, labels)}")


## Experimento 02 — Linear probe sobre backbone congelado

**Hipótese:** a representação do ImageNet já separa bem as raças, mesmo sem nenhuma
adaptação — um classificador linear deve superar o Experimento 01 por larga margem.

A célula a seguir recupera as predições do Experimento 02 e de sua ablação sem BatchNorm, calculando as métricas de validação.

In [ ]:
from dogs.evaluate import load_predictions, metrics_from_logits
from dogs.models import LinearProbe

NOME = "E2_linear_probe"

if RETRAIN:
    from dogs.evaluate import log_result, predict, save_predictions
    from dogs.features import load_features, make_feature_loader
    from dogs.train import get_device, load_checkpoint, train_model

    X_train, y_train = load_features(split="train")
    X_val, y_val = load_features(split="val")
    treino_e2 = make_feature_loader(X_train, y_train, shuffle=True)
    validacao_e2 = make_feature_loader(X_val, y_val, shuffle=False)

    for nome_experimento, usar_batchnorm in [
        ("E2_linear_probe", True),
        ("E2_linear_probe_sem_bn", False),
    ]:
        model_e2 = LinearProbe(X_train.shape[1], use_batchnorm=usar_batchnorm)
        saida_sanity = model_e2(torch.randn(4, X_train.shape[1]))
        assert saida_sanity.shape == (4, 120)

        config_e2 = TrainConfig(
            experiment_name=nome_experimento, num_epochs=30, learning_rate=1e-3
        )
        print(train_model(model_e2, treino_e2, validacao_e2, config_e2))
        model_e2 = load_checkpoint(model_e2, config_e2)
        logits, labels = predict(model_e2, validacao_e2, get_device())
        metricas_e2 = metrics_from_logits(logits, labels)
        log_result(nome_experimento, "val", metricas_e2)
        save_predictions(nome_experimento, "val", logits, labels)
        rotulo = "Experimento 02" if usar_batchnorm else "Experimento 02 sem BatchNorm"
        print(f"{rotulo}: {metricas_e2}")
else:
    for nome_experimento, rotulo in [
        (NOME, "Experimento 02"),
        ("E2_linear_probe_sem_bn", "Experimento 02 sem BatchNorm"),
    ]:
        logits, labels = load_predictions(nome_experimento, "val")
        print(f"{rotulo}: {metrics_from_logits(logits, labels)}")


## Experimento 03 — Fine-tuning parcial

**Hipótese:** descongelar os blocos finais permite adaptar as features de alto nível
ao domínio e deve superar o Experimento 02.

A célula a seguir recupera as predições do Experimento 03 e calcula as métricas de validação e teste.

In [ ]:
import json

from dogs.config import REPORTS_DIR, TrainConfig
from dogs.data import load_data
from dogs.evaluate import (
    load_predictions,
    log_result,
    metrics_from_logits,
    predict,
    save_predictions,
)
from dogs.models import build_finetune_model
from dogs.train import get_device, load_checkpoint, train_model

config = TrainConfig(
    experiment_name="E3_finetune_last2",
    unfreeze_last_n_blocks=2,
    learning_rate=1e-4,
    num_epochs=15,
)

if RETRAIN_E3:
    data = load_data(config)
    class_names = data.class_names

    model = build_finetune_model("resnet50", config.unfreeze_last_n_blocks)
    train_model(model, data.train_loader, data.val_loader, config)
    model = load_checkpoint(model, config)

    device = get_device()
    logits_val, labels_val = predict(model, data.val_loader, device)
    logits_test, labels_test = predict(model, data.test_loader, device)
    metrics_val = metrics_from_logits(logits_val, labels_val)
    metrics_test = metrics_from_logits(logits_test, labels_test)

    for split, logits, labels, metrics in [
        ("val", logits_val, labels_val, metrics_val),
        ("test", logits_test, labels_test, metrics_test),
    ]:
        log_result(config.experiment_name, split, metrics)
        save_predictions(config.experiment_name, split, logits, labels)
else:
    class_names = json.loads(
        (REPORTS_DIR / "class_names.json").read_text(encoding="utf-8")
    )

    logits_val, labels_val = load_predictions(config.experiment_name, "val")
    logits_test, labels_test = load_predictions(config.experiment_name, "test")
    metrics_val = metrics_from_logits(logits_val, labels_val)
    metrics_test = metrics_from_logits(logits_test, labels_test)

    if config.checkpoint_path().exists():
        model = build_finetune_model("resnet50", config.unfreeze_last_n_blocks)
        model = load_checkpoint(model, config)

print(f"Experimento 03 — validação: {metrics_val}")
print(f"Experimento 03 — teste: {metrics_test}")

---
# 5. Resultados

A Tabela 1 reúne os resultados. A comparação principal é feita na validação, o único split disponível para todos os experimentos. A CNN treinada do zero (Experimento 01) alcançou 10,39% de top-1, enquanto o linear probe com embeddings congelados da ResNet50 (Experimento 02) atingiu 88,39%; portanto, transferir a representação aumentou a acurácia em 78,50 pontos percentuais. A ablação sem BatchNorm foi ainda melhor, com 90,89% de top-1, 99,22% de top-5 e F1 macro de 90,56%.

O Experimento 03 de fine-tuning parcial atingiu 85,33% na validação e 85,17% no teste. A diferença de apenas 0,16 ponto percentual entre os dois splits sugere que esse modelo não se ajustou excessivamente à validação. Contudo, neste protocolo o melhor resultado de validação foi o Experimento 02 sem BatchNorm, e não o fine-tuning. A Figura 5 torna esse contraste direto: a transferência de representação explica quase todo o ganho sobre a CNN treinada do zero, enquanto adaptar parcialmente o backbone não trouxe melhoria adicional nesta execução.

A célula a seguir organiza as métricas em uma tabela comparativa.

In [ ]:
results = pd.read_csv(RESULTS_CSV)
nomes_exibicao = {
    "E1_scratch": "Experimento 01",
    "E2_linear_probe": "Experimento 02",
    "E2_linear_probe_sem_bn": "Experimento 02 sem BatchNorm",
    "E3_finetune_last2": "Experimento 03",
}
results_exibicao = results.assign(experiment=results["experiment"].replace(nomes_exibicao))
results_exibicao.sort_values("top1", ascending=False)

### Comparação visual

A célula a seguir apresenta, em gráfico de barras, a acurácia top-1 de validação dos experimentos.

In [ ]:
from dogs.config import FIGURES_DIR
from dogs.viz import comparar_experimentos

comparar_experimentos(
    results_exibicao[results_exibicao["split"] == "val"],
    salvar_em=FIGURES_DIR / "comparacao_top1.png",
)


---
# 6. Análise

## 6.1 O salto do transfer learning

O resultado mais expressivo é a distância entre os Experimentos 01 e 02. A CNN pequena treinada inteiramente com o Stanford Dogs chegou a 10,39% de top-1, enquanto uma única camada linear sobre embeddings congelados chegou a 88,39%, um salto de 78,50 pontos percentuais. A representação transferida, portanto, foi muito mais determinante do que aumentar a capacidade do classificador treinado do zero. O Experimento 03 também superou o baseline com folga (85,33%), mas ficou 3,06 pontos abaixo do Experimento 02 com BatchNorm e 5,56 abaixo da ablação sem BatchNorm.

A ablação contrariou a hipótese inicial: retirar o BatchNorm elevou top-1 de 88,39% para 90,89% e F1 macro de 88,05% para 90,56%. O resultado não mostra que BatchNorm seja prejudicial em geral; nesta configuração específica, os embeddings da ResNet50 já eram suficientemente separáveis e a normalização por batch não acrescentou ganho. Como cada variante foi executada uma única vez, a conclusão correta é que a melhor configuração observada foi a linear sem BatchNorm, e não uma regra universal sobre a camada.

## 6.2 Quais raças o modelo confunde

Os erros se concentram em raças morfologicamente próximas. O recorte da matriz de confusão mostra, por exemplo, confusões entre Siberian husky e Eskimo dog, Lhasa e Shih-Tzu, Staffordshire bullterrier e American Staffordshire terrier, Australian terrier e silky terrier, além dos poodles miniatura e toy. São pares visualmente plausíveis: porte, pelagem, formato da cabeça e origem funcional são semelhantes o bastante para induzir dúvida mesmo em uma inspeção humana rápida.

A aposta feita antes do treino, na Figura 4, acertou o padrão de dificuldade, ainda que não todos os pares exatos. Husky e malamute foram escolhidos como exemplos de cães nórdicos semelhantes; a matriz revelou Husky e Eskimo dog. Também antecipamos confusões entre terriers e entre corgis, e os erros reais confirmaram que classes de aparência muito próxima são a principal fonte de ambiguidade. As Figuras 6 a 8 complementam a matriz ao mostrar pares confundidos, exemplos de maior perda e o recorte normalizado da confusão.

## 6.3 ⚠️ Contaminação entre Stanford Dogs e ImageNet

Há uma limitação estrutural importante: Stanford Dogs foi montado a partir de imagens e anotações do ImageNet, enquanto a ResNet50 usada nos Experimentos 02 e 03 foi pré-treinada no ImageNet-1k. Assim, o backbone pode já ter visto imagens — ou exemplos extremamente próximos — durante o pré-treino. Os 85% a 91% dos modelos transferidos são, portanto, otimistas e não devem ser interpretados como previsão direta de desempenho em fotos novas de cães fora desse domínio.

Essa contaminação não invalida a comparação relativa entre os Experimentos 01, 02 e 03, pois todos foram avaliados no mesmo protocolo; ela limita a generalização externa dos valores absolutos. O Experimento 01 é o único número livre dessa transferência. Um experimento futuro poderia medir o efeito com um conjunto externo de fotografias de raças, sem sobreposição com o ImageNet, e repetir a comparação dos mesmos classificadores.

## 6.4 Limitações

O trabalho foi limitado pelo orçamento computacional, que restringiu o número de épocas e impediu uma busca extensa de hiperparâmetros. Cada experimento foi executado uma única vez, sem múltiplas sementes ou barras de erro, e o teste foi reservado a uma única avaliação do Experimento 03. Também não investigamos outros backbones, métodos especializados de atenção por partes, ensembles, localização do animal ou implantação do modelo. Essas escolhas preservam um protocolo enxuto, mas deixam espaço claro para avaliação mais robusta.

A célula a seguir identifica os pares de raças mais confundidos e apresenta a matriz de confusão recortada, além dos erros mais difíceis.

In [ ]:
from dogs.config import FIGURES_DIR
from dogs.evaluate import most_confused_pairs
from dogs.viz import (
    grid_pares_confundidos,
    grid_piores_erros,
    matriz_confusao_recorte,
    nome_legivel,
)

pares = most_confused_pairs(logits_test, labels_test, class_names, top_n=10)
for real, previsto, contagem in pares:
    print(f"{nome_legivel(real):25s} -> {nome_legivel(previsto):25s} {contagem}x")

nomes_unicos = list(
    dict.fromkeys(nome for real, previsto, _ in pares[:5] for nome in (real, previsto))
)
indices_classes = [class_names.index(nome) for nome in nomes_unicos]

matriz_confusao_recorte(
    logits_test,
    labels_test,
    class_names,
    indices_classes,
    salvar_em=FIGURES_DIR / "matriz_confusao_recorte.png",
)

if RETRAIN:
    from dogs.data import _load_raw

    test_dataset = _load_raw()["test"]
    grid_pares_confundidos(
        test_dataset,
        logits_test,
        labels_test,
        class_names,
        top_n=5,
        salvar_em=FIGURES_DIR / "pares_confundidos.png",
    )
    grid_piores_erros(
        test_dataset,
        logits_test,
        labels_test,
        class_names,
        top_n=10,
        salvar_em=FIGURES_DIR / "piores_erros.png",
    )
else:
    from IPython.display import Image, display

    for nome in ["pares_confundidos", "piores_erros"]:
        caminho = FIGURES_DIR / f"{nome}.png"
        if not caminho.exists():
            raise FileNotFoundError(
                f"Figura da análise ausente: {caminho.name}."
            )
        display(Image(filename=str(caminho)))

---
# 7. Conclusões

A resposta à pergunta central é direta: com dados limitados para uma tarefa fine-grained, transferir uma representação visual é muito mais importante do que aprendê-la do zero. O Experimento 01 chegou a 10,39% de top-1, enquanto o classificador linear do Experimento 02 atingiu 88,39% e sua variante sem BatchNorm chegou a 90,89%. O resultado mais surpreendente foi justamente essa última configuração superar o fine-tuning parcial do Experimento 03, que obteve 85,33% na validação.

Os resultados também precisam ser lidos com cautela: Stanford Dogs deriva do ImageNet, o que torna otimistas as métricas dos modelos transferidos. Ainda assim, o contraste com o Experimento 01 é forte e responde à pergunta experimental no protocolo adotado. O Experimento 03 apresentou valores quase idênticos em validação e teste (85,33% e 85,17%), reforçando que sua diferença para o Experimento 02 não decorre de sobreajuste evidente à validação.

Com mais tempo, compararíamos um segundo backbone, como EfficientNet ou ViT-B/16, sobre os mesmos embeddings; testaríamos CLIP em zero-shot; faríamos ablação de augmentation e repetição com múltiplas sementes; e investigaríamos métodos fine-grained com atenção por partes. Permanecem fora de escopo detecção e localização do cão, ensembles, busca extensiva de hiperparâmetros e deploy.

---
## Referências

- Khosla et al. (2011). *Novel Dataset for Fine-Grained Image Categorization: Stanford Dogs.*
- He et al. (2016). *Deep Residual Learning for Image Recognition.*
- Radford et al. (2021). *Learning Transferable Visual Models From Natural Language Supervision.*